# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Comparación final

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

Balanced accuracy pondera por igual el reconocimiento de daño y de seguro bajo desbalance [1]. Las curvas precisión–recall siguen siendo salvaguardas informativas [2], pero no se suman a BA con pesos arbitrarios: el aprendizaje multiobjetivo recomienda explicitar preferencias y soluciones Pareto [3]. La calibración se audita [4], la revisión se informa con riesgo–cobertura [5], y test permanece fuera de toda selección para evitar sesgo [6]. La capacidad humana, el margen de no inferioridad y los costos son decisiones locales que deben predeclararse.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Entorno recomendado: Google Colab web

Abra una copia nueva desde [GitHub en Google Colab web](https://colab.research.google.com/github/lkoc/Trabajo_PLN-MIA-Grupo4/blob/main/flujo/03_entrenamiento/03_07_comparacion_final.ipynb), seleccione un runtime **CPU** y ejecute desde la primera celda; esta comparación funciona con runtime CPU. Para esta comparación no use el kernel local ni Colab desde VS Code: los candidatos están publicados únicamente en Google Drive y la autorización integrada de `drive.mount()` es más confiable en la interfaz web.

Conserve `COLAB_BUNDLE_SOURCE='github'`. El bootstrap descarga el bundle fijado, verifica todos sus SHA-256, monta Drive y restaura las publicaciones; no solicita seleccionar nueve archivos y no requiere Google Drive para escritorio. `local_upload` queda únicamente como recuperación excepcional si GitHub no contuviera el bundle esperado.

In [ ]:
# Backend reproducible: local o Google Colab desde VS Code
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
import uuid
import zipfile

COLAB_NOTEBOOK_ID = "03_07"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = False
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_AUTO_PUBLISH_MISSING_BUNDLE = True
COLAB_BUNDLE_SOURCE = "github"  # "github" o "local_upload"
COLAB_GITHUB_REPOSITORY = "lkoc/Trabajo_PLN-MIA-Grupo4"
COLAB_GITHUB_REF = "main"
COLAB_GITHUB_BUNDLE_PATH = "resultados/colab_bundle"
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "1ebd741ac91009b92616c6bfc6af47c396396735e5c8f06eb2ec767ef358de9e"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "f3f1c17d77dfa2ca5ebddc7e80f35e35c3f2f5b2f11edfcf955c1a07b418be4a"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _verify_expected_bundle(bundle_dir, expected_bundle_id=COLAB_NOTEBOOK_BUILD_BUNDLE_ID):
    manifest_path = Path(bundle_dir) / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta {manifest_path}")
    manifest = _read_manifest(manifest_path)
    computed_bundle_id = _bundle_id_for_manifest(manifest)
    if manifest.get("bundle_id") != computed_bundle_id:
        raise ValueError("bundle_manifest.json no contiene una identidad válida")
    if computed_bundle_id != expected_bundle_id:
        raise ValueError(
            f"Bundle inesperado: esperado={expected_bundle_id}, obtenido={computed_bundle_id}"
        )
    if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
        raise ValueError("El core del bundle no coincide con el fijado por este cuaderno")
    for name, expected_sha256 in _bundle_specs(manifest):
        artifact = Path(bundle_dir) / name
        if not artifact.is_file() or _sha256(artifact) != expected_sha256:
            raise ValueError(f"Artefacto ausente o inválido: {artifact}")
    return manifest

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if Path(manifest_path) != Path(bundle_dir) / "bundle_manifest.json":
        return False
    try:
        _verify_expected_bundle(bundle_dir, expected_bundle_id)
        return True
    except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _download_bundle_file(url, destination):
    destination = Path(destination)
    partial = destination.with_name(f".{destination.name}.partial")
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "ModeracionPeru-Colab-Bundle/2.0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response, partial.open("wb") as target:
            while block := response.read(1024 * 1024):
                target.write(block)
        os.replace(partial, destination)
    finally:
        if partial.exists():
            partial.unlink()

def _prepare_bundle_staging():
    staging = Path("/content/moderacion_peru_bundle_source")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    return staging

def _uploaded_bundle_member(uploaded, expected_name):
    # Resuelve el nombre exacto o el sufijo (N) que agrega Colab al repetir una carga.
    if expected_name in uploaded:
        return expected_name
    suffix = Path(expected_name).suffix
    base = expected_name[:-len(suffix)] if suffix else expected_name
    prefix = f"{base} ("
    ending = f"){suffix}"
    candidates = []
    for actual_name in uploaded:
        if not actual_name.startswith(prefix) or not actual_name.endswith(ending):
            continue
        duplicate_number = actual_name[len(prefix):-len(ending)]
        if duplicate_number.isdigit():
            candidates.append(actual_name)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError(
            f"La selección contiene varias copias de {expected_name}: {sorted(candidates)}"
        )
    return None

def _acquire_expected_bundle():
    staging = _prepare_bundle_staging()
    if COLAB_BUNDLE_SOURCE == "github":
        encoded_ref = urllib.parse.quote(COLAB_GITHUB_REF, safe="")
        base = (
            f"https://raw.githubusercontent.com/{COLAB_GITHUB_REPOSITORY}/"
            f"{encoded_ref}/{COLAB_GITHUB_BUNDLE_PATH}"
        )
        cache_key = urllib.parse.quote(COLAB_NOTEBOOK_BUILD_BUNDLE_ID, safe="")
        manifest_path = staging / "bundle_manifest.json"
        _download_bundle_file(
            f"{base}/bundle_manifest.json?bundle_id={cache_key}", manifest_path
        )
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            raise ValueError("El manifiesto descargado desde GitHub no es válido")
        if manifest["bundle_id"] != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError(
                "GitHub todavía no contiene el bundle fijado por este cuaderno. "
                "Sincronice resultados/colab_bundle o use COLAB_BUNDLE_SOURCE='local_upload'."
            )
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            raise RuntimeError("GitHub contiene un project_core.zip distinto al esperado")
        for name, _ in _bundle_specs(manifest):
            encoded_name = urllib.parse.quote(name, safe="")
            _download_bundle_file(
                f"{base}/{encoded_name}?bundle_id={cache_key}", staging / name
            )
    elif COLAB_BUNDLE_SOURCE == "local_upload":
        from google.colab import files

        uploaded = files.upload()
        manifest_upload = _uploaded_bundle_member(uploaded, "bundle_manifest.json")
        if manifest_upload is None:
            raise FileNotFoundError("La selección no incluyó bundle_manifest.json")
        (staging / "bundle_manifest.json").write_bytes(uploaded[manifest_upload])
        manifest = _read_manifest(staging / "bundle_manifest.json")
        if manifest.get("bundle_id") != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError("Los archivos seleccionados no pertenecen al bundle esperado")
        required = {"bundle_manifest.json", *(name for name, _ in _bundle_specs(manifest))}
        resolved = {
            name: _uploaded_bundle_member(uploaded, name)
            for name in required - {"bundle_manifest.json"}
        }
        missing = sorted(name for name, actual_name in resolved.items() if actual_name is None)
        if missing:
            raise FileNotFoundError(f"Faltaron archivos del bundle: {missing}")
        for name, actual_name in resolved.items():
            (staging / name).write_bytes(uploaded[actual_name])
    else:
        raise ValueError("COLAB_BUNDLE_SOURCE debe ser 'github' o 'local_upload'")
    return staging, _verify_expected_bundle(staging)

def _write_latest_pointer(releases_dir, release_dir, manifest):
    pointer = {
        "schema_version": "1.0.0",
        "bundle_id": manifest["bundle_id"],
        "core_sha256": manifest["core"]["sha256"],
        "manifest_sha256": _sha256(Path(release_dir) / "bundle_manifest.json"),
        "published_at": datetime.now(timezone.utc).isoformat(),
    }
    latest_path = Path(releases_dir) / "latest.json"
    partial = Path(releases_dir) / f".latest-{uuid.uuid4().hex}.json"
    partial.write_text(json.dumps(pointer, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(partial, latest_path)
    return pointer

def _publish_expected_bundle(staging, releases_dir):
    manifest = _verify_expected_bundle(staging)
    releases_dir = Path(releases_dir)
    releases_dir.mkdir(parents=True, exist_ok=True)
    release_dir = releases_dir / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if release_dir.exists():
        _verify_expected_bundle(release_dir)
        release_status = "already_present_and_verified"
    else:
        partial = releases_dir / f".{COLAB_NOTEBOOK_BUILD_BUNDLE_ID}.partial-{uuid.uuid4().hex}"
        partial.mkdir()
        try:
            for name, _ in _bundle_specs(manifest):
                shutil.copyfile(Path(staging) / name, partial / name)
            shutil.copyfile(
                Path(staging) / "bundle_manifest.json",
                partial / "bundle_manifest.json",
            )
            _verify_expected_bundle(partial)
            os.replace(partial, release_dir)
        finally:
            if partial.exists():
                shutil.rmtree(partial)
        release_status = "auto_published_and_verified"
    pointer = _write_latest_pointer(releases_dir, release_dir, manifest)
    return {
        "status": release_status,
        "release_dir": release_dir,
        "latest_pointer": pointer,
    }

def _ensure_expected_drive_release(releases_dir):
    release_dir = Path(releases_dir) / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if _bundle_is_current(
        release_dir,
        release_dir / "bundle_manifest.json",
        COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
    ):
        return {"status": "already_present_and_verified", "release_dir": release_dir}
    if not COLAB_AUTO_PUBLISH_MISSING_BUNDLE:
        raise RuntimeError(
            "Drive no contiene el release esperado y COLAB_AUTO_PUBLISH_MISSING_BUNDLE=False"
        )
    staging, _ = _acquire_expected_bundle()
    return _publish_expected_bundle(staging, releases_dir)

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    RELEASES_DIR.mkdir(parents=True, exist_ok=True)
    release_check = _ensure_expected_drive_release(RELEASES_DIR)
    latest_bundle_id = COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    RELEASE_DIR = RELEASES_DIR / latest_bundle_id
    manifest = _verify_expected_bundle(RELEASE_DIR)
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    latest_pointer = _read_manifest(latest_pointer_path) if latest_pointer_path.is_file() else {}
    latest_matches_notebook = (
        latest_pointer.get("bundle_id") == COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        and latest_pointer.get("core_sha256") == COLAB_EXPECTED_CORE_SHA256
        and latest_pointer.get("manifest_sha256") == _sha256(release_manifest_path)
    )
    if latest_matches_notebook:
        release_source = (
            "auto_published_from_" + COLAB_BUNDLE_SOURCE
            if release_check["status"] == "auto_published_and_verified"
            else "latest_pointer"
        )
    else:
        # Un cuaderno reproducible puede activar su release inmutable exacto aunque
        # latest todavía apunte a otra versión; jamás mezcla código e inputs.
        release_source = "notebook_pinned_release"
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive después de "
                f"verificar o autopublicar {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'origen_del_release': release_source,
        'estado_del_release': release_check['status'],
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Prompt operacional vigente', {'ruta': OPERATIONAL_PROMPT, 'versión': '3.2.0'}, tone='success')


## Procedimiento reproducible por corridas

**Entorno recomendado.** Abra una copia nueva de este cuaderno desde GitHub en **Google Colab web**, seleccione un runtime **CPU** y ejecute desde la primera celda. No use para esta corrida el kernel local ni Colab desde VS Code: los candidatos están publicados únicamente en Google Drive y la autorización integrada de `drive.mount()` es más confiable en la interfaz web. Mantenga `COLAB_BUNDLE_SOURCE='github'`; el bundle se descarga y verifica automáticamente, sin selector manual de archivos ni Google Drive para escritorio.

Este cuaderno se ejecuta por compuertas deliberadas. Use el mismo `COLAB_RUN_ID='03_07_working_v2_1'` para que la selección congelada se restaure desde Drive. Nunca active comparación y test en la misma corrida.

1. **Inicio sin carga manual.** Abra <https://colab.research.google.com/github/lkoc/Trabajo_PLN-MIA-Grupo4/blob/main/flujo/03_entrenamiento/03_07_comparacion_final.ipynb>, confirme runtime CPU y conserve `COLAB_BUNDLE_SOURCE='github'`. `local_upload` queda solo como recuperación excepcional; la corrida normal no solicita nueve archivos.
2. **Preflight de candidatos.** Mantenga `RUN_COMPARE_AND_FREEZE=False`, `RUN_TEST_ONCE=False` y `RUN_PUBLISH=False`. La restauración verifica los manifiestos y SHA-256 de `03_01`–`03_06`; `03_06b` solo se agrega si existe y es elegible. No avance hasta ver `listo_para_comparar=True`.
3. **Predeclaración.** Fije `MAX_REVIEW_RATE` y `MACRO_AUPRC_NONINFERIORITY_MARGIN` antes de comparar. Si permanecen en `None`, se genera el informe/Pareto, pero test sigue bloqueado.
4. **Comparación en validation.** Active solo `RUN_COMPARE_AND_FREEZE=True`. Mantenga test y publicación en `False`. El resultado y la selección congelada se guardan como checkpoint verificable del run `03_07` en Drive.
5. **Revisión humana del congelado.** Vuelva a dejar la comparación en `False` y revise candidatos, umbrales, capacidad, margen y advertencias. No cambie estos valores después de mirar test.
6. **Apertura única de test.** Solo con la selección aprobada active `RUN_TEST_ONCE=True`. Si el modelo elegido necesita GPU, cambie el runtime pero conserve el mismo run y ejecute de nuevo desde arriba. Test se infiere una vez y se vuelve a guardar en Drive.
7. **Publicación.** `RUN_PUBLISH` permanece en `False`; la publicación productiva requiere una aprobación posterior separada.

Si una sesión se desconecta antes de terminar el bootstrap o la comparación, reanude desde la primera celda. No borre el run ni copie manualmente candidatos entre snapshots.

## Restauración reproducible del dataset

In [ ]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


## Restauración verificable de candidatos desde Drive

In [ ]:
from pathlib import Path
import shutil
import time
from moderacion_peru.colab import restore_colab_run_outputs
from moderacion_peru.ensemble_evaluation import audit_validation_candidate_eligibility

DATA=COLAB_CONTEXT.input('dataset_5_salidas') if COLAB_CONTEXT else ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
LOCAL_CANDIDATE_ROOT=ROOT/'modelos/v2'
# Si usa Google Drive para escritorio, indique aquí la carpeta ModeracionPeru_Colab.
# En un kernel Colab se monta y detecta automáticamente; no requiere Drive Desktop.
LOCAL_GOOGLE_DRIVE_ROOT=None  # p. ej. Path('G:/My Drive/ModeracionPeru_Colab')
RESTORE_CANDIDATES_FROM_DRIVE=True
DRIVE_RUN_IDS={
    '03_01':'03_01_working_v2_1',
    '03_02':'03_02_working_v2_1',
    '03_03':'03_03_working_v2_1',
    '03_03b':'03_03b_working_v2_1',
    '03_04':'03_04_working_v2_1',
    '03_05':'03_05_working_v2_1',
    '03_06':'03_06_working_v2_1',
    '03_06b':'03_06b_working_v2_1',
}
OPTIONAL_DRIVE_RUNS={'03_06b'}
REQUIRED_FAMILIES={
    '03_01':('classical:',),
    '03_02':('flat_minilm','flat_e5'),
    '03_03':('cascade',),
    '03_03b':('cascade_v2',),
    '03_04':('multitask',),
    '03_05':('qwen_lora',),
    '03_06':('qwen_structured',),
}

drive_candidate_root=(
    COLAB_CONTEXT.runtime_root/'comparison_inputs'/'03_07'
    if COLAB_CONTEXT is not None
    else ROOT/'modelos/v2/restored_from_drive'
)
drive_root=(
    COLAB_CONTEXT.drive_root
    if COLAB_CONTEXT is not None
    else Path(LOCAL_GOOGLE_DRIVE_ROOT).expanduser()
    if LOCAL_GOOGLE_DRIVE_ROOT is not None
    else None
)
CANDIDATE_ROOTS=[LOCAL_CANDIDATE_ROOT]
restoration=[]
if RESTORE_CANDIDATES_FROM_DRIVE and drive_root is not None:
    for restore_index,(notebook_id,run_id) in enumerate(DRIVE_RUN_IDS.items(),start=1):
        destination=drive_candidate_root/notebook_id/run_id
        show_callout(
            f'Restauración {restore_index}/{len(DRIVE_RUN_IDS)} · {notebook_id}',
            f'Verificando SHA-256 y extrayendo {run_id}. Esta etapa puede tardar varios minutos.',
            tone='info',
        )
        restore_started=time.perf_counter()
        try:
            restored=restore_colab_run_outputs(drive_root,notebook_id=notebook_id,run_id=run_id,destination=destination)
            restore_entry={'notebook_id':notebook_id,'run_id':run_id,**restored}
        except (FileNotFoundError,ValueError) as exc:
            corrupt=isinstance(exc,ValueError)
            restore_entry={
                'notebook_id':notebook_id,
                'run_id':run_id,
                'status':(
                    'corrupt_optional' if corrupt and notebook_id in OPTIONAL_DRIVE_RUNS else
                    'corrupt_required' if corrupt else
                    'missing_optional' if notebook_id in OPTIONAL_DRIVE_RUNS else
                    'missing_required'
                ),
                'source':str(Path(drive_root)/'runs'/notebook_id/run_id),
                'detail':str(exc),
            }
        restoration.append(restore_entry)
        disk=shutil.disk_usage(drive_candidate_root)
        show_result(f'Estado {restore_index}/{len(DRIVE_RUN_IDS)} · {notebook_id}',{
            'status':restore_entry['status'],
            'minutos':round((time.perf_counter()-restore_started)/60,2),
            'disco_usado_GiB':round((disk.total-disk.free)/(1024**3),2),
            'disco_libre_GiB':round(disk.free/(1024**3),2),
            'detalle':restore_entry.get('detail'),
        },tone='success' if restore_entry['status'].startswith(('restored','existing')) else 'warning')
    CANDIDATE_ROOTS.append(drive_candidate_root)
elif RESTORE_CANDIDATES_FROM_DRIVE:
    show_callout(
        'Drive no está montado en el kernel local',
        'Seleccione un kernel Google Colab y ejecute desde la primera celda, o configure LOCAL_GOOGLE_DRIVE_ROOT si usa Drive para escritorio.',
        tone='warning',
    )

candidate_audit=audit_validation_candidate_eligibility(DATA,CANDIDATE_ROOTS)
eligible_candidates=[
    {
        'candidate_id':row.get('candidate_id'),
        'model_family':row.get('model_family'),
        'training_regime':row.get('training_regime','full'),
        'candidate_path':row.get('candidate_path'),
    }
    for row in candidate_audit['eligible']
]
eligible_families={str(row.get('model_family','')).casefold() for row in candidate_audit['eligible']}

def family_is_present(expected_family):
    expected=str(expected_family).casefold()
    return any(
        family.startswith(expected) if expected.endswith(':') else family==expected
        for family in eligible_families
    )

missing_required_families={
    notebook_id:[family for family in families if not family_is_present(family)]
    for notebook_id,families in REQUIRED_FAMILIES.items()
}
missing_required_families={key:value for key,value in missing_required_families.items() if value}
prompt_sft_active=family_is_present('qwen_prompt_sft')
required_restore_failures=[
    row for row in restoration
    if row.get('status') in {'missing_required','corrupt_required'}
]
CANDIDATE_PREFLIGHT_READY=(
    candidate_audit['eligible_count']>0
    and not missing_required_families
    and not required_restore_failures
)
show_result('Restauración de publicaciones 03_01–03_06b',{
    'drive_root':drive_root,
    'runs':restoration,
    '03_06b':'incluido' if prompt_sft_active else 'no encontrado o no elegible; omitido',
},tone='success' if CANDIDATE_PREFLIGHT_READY else 'warning')
show_result('Elegibilidad previa a 03_07',{
    'dataset_sha256':candidate_audit['dataset_sha256'],
    'descubiertos':candidate_audit['discovered_count'],
    'elegibles':eligible_candidates,
    'rechazados':candidate_audit['rejected'],
    'publicaciones_requeridas_fallidas':required_restore_failures,
    'familias_requeridas_ausentes':missing_required_families,
    'listo_para_comparar':CANDIDATE_PREFLIGHT_READY,
},tone='success' if CANDIDATE_PREFLIGHT_READY else 'warning')
if prompt_sft_active:
    show_callout('03_06b activado','Existe un candidato completo, con validation común y test sellado; entrará en la comparación.',tone='success')
else:
    show_callout('03_06b omitido','Su ausencia o inelegibilidad no bloquea la comparación de 03_01–03_06.',tone='neutral')

## Configuración y ejecución

In [ ]:
from moderacion_peru.ensemble_evaluation import compare_and_freeze_validation,evaluate_frozen_test
RESULT_ROOT=(COLAB_CONTEXT.scratch_output_dir/'resultados_modelos' if COLAB_CONTEXT else ROOT/'resultados/modelos')
COMPARISON=RESULT_ROOT/'comparacion_individual_ensemble_validation.json'
FREEZE=RESULT_ROOT/'seleccion_congelada.json'
TEST_REPORT=RESULT_ROOT/'test_final_abierto_una_vez.json'
PARALLEL_WORKERS=4  # bootstrap pareado por video
BOOTSTRAP_REPLICATES=2000
SELECTION_FOLDS=5
# Deben acordarse ANTES de comparar. None permite el informe/Pareto, pero mantiene test sellado.
MAX_REVIEW_RATE=None  # p. ej. 0.10 solo si capacidad humana <=10% fue aprobada
MACRO_AUPRC_NONINFERIORITY_MARGIN=None  # p. ej. 0.02 solo si fue predeclarado
RUN_COMPARE_AND_FREEZE=False
RUN_TEST_ONCE=False
RUN_PUBLISH=False

def checkpoint_03_07_to_drive(label):
    if COLAB_CONTEXT is None:
        return None
    from moderacion_peru.colab import publish_colab_outputs
    publication=publish_colab_outputs(COLAB_CONTEXT)
    show_result(label,publication,tone='success')
    return publication

if RUN_COMPARE_AND_FREEZE:
    if not CANDIDATE_PREFLIGHT_READY:
        raise RuntimeError(
            'Preflight incompleto: faltan familias requeridas de 03_01–03_06. '
            f'Revise familias_requeridas_ausentes={missing_required_families} y los run_id de Drive.'
        )
    comparison_result=run_with_progress('BA OOF, riesgo-cobertura y bootstrap',compare_and_freeze_validation,DATA,CANDIDATE_ROOTS,COMPARISON,FREEZE,bootstrap_replicates=BOOTSTRAP_REPLICATES,selection_folds=SELECTION_FOLDS,max_review_rate=MAX_REVIEW_RATE,macro_auprc_noninferiority_margin=MACRO_AUPRC_NONINFERIORITY_MARGIN,parallel_workers=PARALLEL_WORKERS,progress_unit='réplica')
    show_result('Comparación y congelación en validation',comparison_result,tone='success')
    checkpoint_03_07_to_drive('Checkpoint verificable de la comparación en Drive')
if RUN_TEST_ONCE:
    if not FREEZE.is_file():
        raise FileNotFoundError('Falta la selección congelada; ejecute y revise primero RUN_COMPARE_AND_FREEZE.')
    test_result=run_with_progress('Inferencia de test',evaluate_frozen_test,FREEZE,TEST_REPORT,confirm_single_test_open=True,progress_unit='lote')
    show_result('Apertura única de test natural + vista 4:1',test_result,tone='warning')
    checkpoint_03_07_to_drive('Checkpoint verificable del test abierto una vez')
if RUN_PUBLISH:
    raise RuntimeError('Publicación productiva bloqueada por diseño: habilítela solo tras aprobación posterior y revisión de FREEZE/TEST_REPORT.')
if not (RUN_COMPARE_AND_FREEZE or RUN_TEST_ONCE or RUN_PUBLISH):
    show_summary('Criterio vigente',{'candidatos_elegibles':candidate_audit['eligible_count'],'03_06b':'incluido' if prompt_sft_active else 'omitido','preflight_listo':CANDIDATE_PREFLIGHT_READY,'ranking':'BA binaria ANY_DAMAGE OOF a cobertura completa','agregación':'lexicográfica; no suma métricas redundantes','salvaguarda':'macro-AUPRC daños + frontera Pareto','desempate':'menor R_0.67; luego macro-AUPRC','NEEDS_REVIEW':'política posterior bajo capacidad humana declarada','bootstrap':f'{BOOTSTRAP_REPLICATES} réplicas pareadas por video en {PARALLEL_WORKERS} hilos','persistencia':'validation se publica como checkpoint verificable del run 03_07 en Drive','test':'bloqueado hasta fijar capacidad y margen antes de comparar'},tone='neutral')

## Referencias

[1] K. H. Brodersen, C. S. Ong, K. E. Stephan, et al., "The Balanced Accuracy and Its Posterior Distribution," in Proc. 20th Int. Conf. Pattern Recognition, 2010, pp. 3121–3124, doi: 10.1109/ICPR.2010.764.

[2] T. Saito and M. Rehmsmeier, "The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets," PLOS ONE, vol. 10, no. 3, Art. no. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[3] Y. Jin and B. Sendhoff, "Pareto-Based Multiobjective Machine Learning: An Overview and Case Studies," IEEE Trans. Syst., Man, Cybern. C, vol. 38, no. 3, pp. 397–415, 2008, doi: 10.1109/TSMCC.2008.919172.

[4] C. Guo, G. Pleiss, Y. Sun, et al., "On Calibration of Modern Neural Networks," in Proc. ICML, vol. 70, 2017, pp. 1321–1330. [Online]. Available: https://proceedings.mlr.press/v70/guo17a.html

[5] Y. Geifman and R. El-Yaniv, "Selective Classification for Deep Neural Networks," in Adv. Neural Inf. Process. Syst., vol. 30, 2017.

[6] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.